# Python for GenAI & Data Analysis
## Logging, Debugging Practices, Unit Testing & Code Quality

This notebook is a hands-on companion for the module on writing production-ready,
maintainable, and testable Python code, with emphasis on patterns used in
GenAI and AI pipeline development.


**Format:** Concept + runnable example + exercise, per section

### Contents
1. Why Code Quality Matters in AI Applications
2. Writing Clean Python Code
3. Python Exceptions
4. Exception Handling
5. Debugging in Python
6. Debugging Tools (VS Code Debugger)
7. Logging
8. Virtual Environments and Dependency Management
9. Code Formatting (PEP 8)
10. Linting
11. Static Type Hints
12. Unit Testing Fundamentals (unittest)
13. Pytest
14. Mocking
15. Testing AI Applications
16. Measuring Code Coverage
17. Performance Profiling
18. Configuration Management
19. Project Structure for AI Applications
20. Mini Hands-on Exercises
21. Capstone Project: Production-Ready Document Processing Pipeline


## 1. Why Code Quality Matters in AI Applications

AI and GenAI applications are especially prone to production failures because they combine
non-deterministic model outputs, external API dependencies, and rapidly changing data.

**Common reasons AI applications fail in production**
- Unhandled API failures (timeouts, rate limits, malformed responses)
- Silent failures caused by broad `except:` blocks
- No logging, so failures cannot be diagnosed after the fact
- Hardcoded configuration and secrets
- No tests, so refactoring breaks existing behavior
- Code written for a single notebook run, never designed for reuse

**Production mindset vs. college assignment mindset**

| College Assignment | Production Code |
|---|---|
| Works once, on one input | Works reliably across many inputs and edge cases |
| No error handling | Explicit, targeted error handling |
| No logs | Structured logs at every meaningful step |
| No tests | Unit tests covering core logic |
| One large script | Modular, reusable components |
| Hardcoded values | Externalized configuration |

The rest of this notebook builds each of these habits from the ground up.


## 2. Writing Clean Python Code

### 2.1 Naming Conventions

| Element | Convention | Example |
|---|---|---|
| Variable | snake_case | `user_name` |
| Function | snake_case, verb-based | `calculate_total()` |
| Class | PascalCase | `DocumentLoader` |
| Constant | UPPER_SNAKE_CASE | `MAX_RETRIES` |


In [1]:
# Naming convention examples

user_name = "aditi"

def calculate_total(price: float, quantity: int) -> float:
    return price * quantity

class DocumentLoader:
    pass

MAX_RETRIES = 3

print(user_name, calculate_total(10.5, 3), MAX_RETRIES)


aditi 31.5 3


### 2.2 Recommended Project File Structure

```
project/
    main.py
    utils.py
    config.py
    models.py
    tests/
    data/
```

- `main.py` — entry point, orchestrates the workflow
- `utils.py` — small reusable helper functions
- `config.py` — configuration and constants
- `models.py` — data models / classes
- `tests/` — all unit tests
- `data/` — sample or working data files


### 2.3 Modular Programming

Breaking logic into small, single-purpose functions makes code easier to test,
reuse, and reason about. Compare the two implementations below.


In [2]:
# Non-modular version: everything in one block

def process_orders_bad(orders):
    total = 0
    for order in orders:
        if order["quantity"] > 0 and order["price"] > 0:
            line_total = order["quantity"] * order["price"]
            if order.get("discount"):
                line_total = line_total - (line_total * order["discount"])
            total += line_total
    return total


In [3]:
# Modular version: split into single-purpose functions

def is_valid_order(order: dict) -> bool:
    return order.get("quantity", 0) > 0 and order.get("price", 0) > 0

def calculate_line_total(order: dict) -> float:
    line_total = order["quantity"] * order["price"]
    discount = order.get("discount", 0)
    return line_total * (1 - discount)

def process_orders(orders: list) -> float:
    return sum(
        calculate_line_total(order)
        for order in orders
        if is_valid_order(order)
    )

sample_orders = [
    {"quantity": 2, "price": 100, "discount": 0.1},
    {"quantity": 0, "price": 50},
    {"quantity": 1, "price": 200},
]

print(process_orders(sample_orders))


380.0


### 2.4 Comments

**Good comments** explain *why* something is done, not *what* the code already says.

**Bad comments** restate the obvious and go stale as code changes.

```python
# Bad: restates the code
x = x + 1  # increment x by 1

# Good: explains intent
x = x + 1  # account for the header row when indexing
```

Use comments when the *reason* for a decision is not obvious from the code itself
(a workaround, a business rule, a non-obvious edge case).


### 2.5 Docstrings

Docstrings document the *contract* of a function: what it expects and what it returns.
They are readable by humans and by tools (`help()`, IDEs, documentation generators).


In [4]:
def calculate_sum(a: float, b: float) -> float:
    """Return the sum of two numbers.

    Args:
        a: First number.
        b: Second number.

    Returns:
        The sum of a and b.
    """
    return a + b

help(calculate_sum)


Help on function calculate_sum in module __main__:

calculate_sum(a: float, b: float) -> float
    Return the sum of two numbers.

    Args:
        a: First number.
        b: Second number.

    Returns:
        The sum of a and b.



## 3. Python Exceptions

Understanding common built-in exceptions is the first step toward writing code that
fails predictably and informatively.

| Exception | Typical Cause |
|---|---|
| `SyntaxError` | Invalid Python syntax |
| `TypeError` | Operation applied to an incompatible type |
| `ValueError` | Correct type, invalid value |
| `IndexError` | Sequence index out of range |
| `KeyError` | Dictionary key not found |
| `AttributeError` | Attribute/method does not exist on object |
| `ImportError` | Import statement fails |
| `ModuleNotFoundError` | Module cannot be located |
| `FileNotFoundError` | File path does not exist |
| `ZeroDivisionError` | Division by zero |


In [5]:
# Hands-on: triggering common exceptions individually

examples = {}

try:
    1 / 0
except ZeroDivisionError as e:
    examples["ZeroDivisionError"] = str(e)

try:
    int("abc")
except ValueError as e:
    examples["ValueError"] = str(e)

try:
    [1, 2, 3][10]
except IndexError as e:
    examples["IndexError"] = str(e)

try:
    {"a": 1}["b"]
except KeyError as e:
    examples["KeyError"] = str(e)

try:
    "text".non_existent_method()
except AttributeError as e:
    examples["AttributeError"] = str(e)

try:
    open("no_such_file.txt")
except FileNotFoundError as e:
    examples["FileNotFoundError"] = str(e)

for name, message in examples.items():
    print(f"{name}: {message}")


ZeroDivisionError: division by zero
ValueError: invalid literal for int() with base 10: 'abc'
IndexError: list index out of range
KeyError: 'b'
AttributeError: 'str' object has no attribute 'non_existent_method'
FileNotFoundError: [Errno 2] No such file or directory: 'no_such_file.txt'


## 4. Exception Handling

The `try / except / else / finally` structure controls how errors are caught and
how cleanup logic runs regardless of success or failure.

- `try` — code that might raise an exception
- `except` — handles a specific exception type
- `else` — runs only if no exception occurred
- `finally` — always runs, used for cleanup


In [6]:
def read_number_from_string(value: str) -> int:
    try:
        result = int(value)
    except ValueError:
        print(f"Could not convert '{value}' to an integer")
        return None
    else:
        print(f"Conversion succeeded: {result}")
        return result
    finally:
        print("Conversion attempt finished")

read_number_from_string("42")
read_number_from_string("abc")


Conversion succeeded: 42
Conversion attempt finished
Could not convert 'abc' to an integer
Conversion attempt finished


In [7]:
# Handling multiple exception types distinctly

def load_config_value(config: dict, key: str, path: str) -> str:
    try:
        with open(path) as f:
            f.read()
        return config[key]
    except FileNotFoundError:
        return f"Config file not found at {path}"
    except KeyError:
        return f"Key '{key}' not found in config"

print(load_config_value({"model": "gpt-4"}, "model", "missing.txt"))
print(load_config_value({"model": "gpt-4"}, "temperature", "missing.txt"))


Config file not found at missing.txt
Config file not found at missing.txt


### 4.1 Raising Exceptions

Use `raise` to signal that a function cannot continue under the given conditions,
with a message that tells the caller exactly what went wrong.


In [8]:
def validate_document(text: str) -> None:
    if not text or not text.strip():
        raise ValueError("Invalid document: content is empty")

try:
    validate_document("   ")
except ValueError as e:
    print(e)


Invalid document: content is empty


### 4.2 Creating Custom Exceptions

Custom exceptions make error handling specific to your application's domain,
so callers can distinguish your errors from generic Python errors.


In [9]:
class InvalidDocumentError(Exception):
    """Raised when a document fails validation checks."""
    pass

class UnsupportedFileTypeError(Exception):
    """Raised when a file extension is not supported by the pipeline."""
    pass

def validate_document_v2(text: str) -> None:
    if not text or not text.strip():
        raise InvalidDocumentError("Document content is empty")

def check_file_type(filename: str, allowed: tuple) -> None:
    if not filename.lower().endswith(allowed):
        raise UnsupportedFileTypeError(f"'{filename}' has an unsupported extension")

try:
    validate_document_v2("")
except InvalidDocumentError as e:
    print(f"InvalidDocumentError: {e}")

try:
    check_file_type("report.exe", (".txt", ".csv", ".pdf"))
except UnsupportedFileTypeError as e:
    print(f"UnsupportedFileTypeError: {e}")


InvalidDocumentError: Document content is empty
UnsupportedFileTypeError: 'report.exe' has an unsupported extension


## 5. Debugging in Python

### 5.1 Key Terms

- **Bug** — a flaw in the code that causes incorrect behavior
- **Error** — a problem detected while parsing or running code
- **Exception** — a runtime event that interrupts normal program flow

### 5.2 Types of Bugs

| Type | Description |
|---|---|
| Logic Bug | Code runs but produces the wrong result |
| Runtime Bug | Code fails while executing (e.g., unhandled exception) |
| Syntax Bug | Code does not parse |
| Performance Bug | Code is correct but too slow or resource-heavy |
| Memory Bug | Excessive or leaking memory usage |


In [10]:
# Example: a logic bug (runs without error, wrong result)

def average(numbers: list) -> float:
    return sum(numbers) / len(numbers) - 1  # bug: subtracts 1 unintentionally

print(average([2, 4, 6]))  # expected 4.0, actual is wrong


3.0


In [11]:
# Corrected version

def average_fixed(numbers: list) -> float:
    if not numbers:
        raise ValueError("Cannot compute average of an empty list")
    return sum(numbers) / len(numbers)

print(average_fixed([2, 4, 6]))


4.0


### 5.3 Print Debugging

Print debugging is the fastest way to inspect state, but it does not scale to
larger applications and must be removed manually afterward.


In [12]:
def chunk_text(text: str, chunk_size: int) -> list:
    chunks = []
    for i in range(0, len(text), chunk_size):
        chunk = text[i:i + chunk_size]
        print(f"i={i}, chunk={chunk!r}")  # print debugging
        chunks.append(chunk)
    return chunks

chunk_text("the quick brown fox", chunk_size=6)


i=0, chunk='the qu'
i=6, chunk='ick br'
i=12, chunk='own fo'
i=18, chunk='x'


['the qu', 'ick br', 'own fo', 'x']

**Advantages of print debugging:** fast, no setup required, works anywhere.

**Limitations:** clutters output, must be manually removed, no severity levels,
not searchable or persistent, unsuitable for production.


### 5.4 Assertions

Assertions validate assumptions during development. They should never be relied on
for handling expected runtime errors, since they can be disabled with the `-O` flag.


In [13]:
def register_user(age: int) -> str:
    assert age > 0, "Age must be positive"
    assert age < 130, "Age must be realistic"
    return "User registered"

try:
    register_user(-5)
except AssertionError as e:
    print(f"AssertionError: {e}")

print(register_user(28))


AssertionError: Age must be positive
User registered


## 6. Debugging with the VS Code Debugger

Print statements and assertions are useful for quick checks, but a real debugger
lets you pause execution and inspect the full program state.

**Core VS Code debugger controls**
- Setting breakpoints (click left of the line number)
- Step Into (F11) — enter a function call
- Step Over (F10) — execute the current line without entering calls
- Step Out (Shift+F11) — finish the current function and return to the caller
- Continue (F5) — run until the next breakpoint
- Restart / Stop — restart or end the debug session

**Inspection panels**
- Watch — track specific expressions as they change
- Call Stack — see the chain of function calls that led to the current line
- Variables — inspect local and global variable values
- Debug Console — run expressions in the current paused context

**Conditional breakpoints** pause execution only when a specified condition is true,
which is essential when debugging inside loops that run many iterations, or when
debugging nested functions and API response handling where the bug only appears
on a specific input.

**Exercise (do this in VS Code, not in this notebook):**
1. Open `chunk_text` from Section 5.3 in VS Code.
2. Set a breakpoint inside the loop.
3. Add a conditional breakpoint that triggers only when `i > 6`.
4. Run the debugger and inspect `chunk` in the Variables panel at each step.


## 7. Logging

### 7.1 Why Logging Instead of `print()`

**Problems with `print()`**
- No severity levels
- No timestamps or context by default
- Cannot be selectively enabled or disabled
- Cannot be easily routed to files, monitoring systems, or log aggregators

**Benefits of logging**
- Configurable severity levels
- Consistent, structured, timestamped output
- Can log to console, file, or external systems simultaneously
- Essential for server applications and long-running AI pipelines where issues
  must be diagnosed after the fact, not just during a live session

### 7.2 Logging Levels

| Level | Use Case |
|---|---|
| `DEBUG` | Detailed diagnostic information, developer use only |
| `INFO` | Confirmation that things are working as expected |
| `WARNING` | Something unexpected happened, but the program continues |
| `ERROR` | A serious problem prevented a specific operation from completing |
| `CRITICAL` | A severe error that may cause the program to stop |


In [14]:
import logging

# Basic configuration: level, format, and output stream
logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s | %(levelname)s | %(filename)s:%(lineno)d | %(funcName)s | %(message)s",
)

logger = logging.getLogger("genai_pipeline")

logger.debug("Loading configuration from config.yaml")
logger.info("Model initialized successfully")
logger.warning("API response took longer than expected: 4.2s")
logger.error("Failed to parse model response as JSON")
logger.critical("Database connection lost, shutting down pipeline")


ERROR:genai_pipeline:Failed to parse model response as JSON
CRITICAL:genai_pipeline:Database connection lost, shutting down pipeline


### 7.3 Logging to File with Rotation

In production, logs are written to files and rotated so they do not grow without bound.


In [15]:
import logging
from logging.handlers import RotatingFileHandler
import os

os.makedirs("logs", exist_ok=True)

file_logger = logging.getLogger("file_logger")
file_logger.setLevel(logging.INFO)

handler = RotatingFileHandler(
    "logs/application.log",
    maxBytes=1_000_000,   # rotate after ~1 MB
    backupCount=3,        # keep 3 backup files
)
formatter = logging.Formatter(
    "%(asctime)s | %(levelname)s | %(name)s | %(message)s"
)
handler.setFormatter(formatter)
file_logger.addHandler(handler)

file_logger.info("Pipeline started")
file_logger.warning("Retry attempt 1 for embedding API call")

with open("logs/application.log") as f:
    print(f.read())


INFO:file_logger:Pipeline started


2026-08-03 07:46:03,101 | INFO | file_logger | Pipeline started
2026-08-03 07:46:03,105 | WARNING | file_logger | Retry attempt 1 for embedding API call



### 7.4 Logging Best Practices

**Never log:**
- Passwords
- API keys or tokens
- Personally identifiable information (PII)

Log identifiers or redacted values instead of raw secrets.


In [16]:
def call_model_api(api_key: str, prompt: str) -> str:
    masked_key = api_key[:4] + "..." + api_key[-2:]
    file_logger.info(f"Calling model API with key={masked_key}, prompt_length={len(prompt)}")
    return "mock response"

call_model_api("sk-abcdef1234567890", "Summarize this document")


INFO:file_logger:Calling model API with key=sk-a...90, prompt_length=23


'mock response'

### 7.5 Logging in AI Applications

For GenAI pipelines, structured logs should capture operational metrics needed for
debugging, cost tracking, and reliability monitoring:
prompt (or a hash/summary of it), response time, token count, API failures,
retry attempts, latency, model name, and cost estimation.


In [17]:
import time
import random

def call_llm_with_logging(prompt: str, model: str = "gpt-4o-mini") -> dict:
    start_time = time.time()
    file_logger.info(f"model={model} | prompt_chars={len(prompt)} | event=request_start")

    # simulated call
    time.sleep(0.05)
    token_count = len(prompt.split()) * 2
    latency = time.time() - start_time
    cost_estimate = token_count * 0.000002

    file_logger.info(
        f"model={model} | tokens={token_count} | latency_s={latency:.3f} "
        f"| cost_usd={cost_estimate:.6f} | event=request_success"
    )

    return {
        "model": model,
        "tokens": token_count,
        "latency_s": round(latency, 3),
        "cost_usd": round(cost_estimate, 6),
    }

result = call_llm_with_logging("Summarize the quarterly report in three bullet points")
print(result)


INFO:file_logger:model=gpt-4o-mini | prompt_chars=53 | event=request_start
INFO:file_logger:model=gpt-4o-mini | tokens=16 | latency_s=0.051 | cost_usd=0.000032 | event=request_success


{'model': 'gpt-4o-mini', 'tokens': 16, 'latency_s': 0.051, 'cost_usd': 3.2e-05}


## 8. Virtual Environments and Dependency Management

### 8.1 Why Isolated Environments Matter

Every project should have its own virtual environment so dependencies (and their
versions) do not conflict across projects.

```bash
# Create a virtual environment
python -m venv venv

# Activate it
# macOS/Linux
source venv/bin/activate
# Windows
venv\Scripts\activate
```

### 8.2 Dependency Management

```bash
# Install a package
pip install requests

# Update a package
pip install --upgrade requests

# Record exact installed versions
pip freeze > requirements.txt

# Recreate the environment elsewhere
pip install -r requirements.txt
```

Pinning versions in `requirements.txt` ensures the project behaves the same way
across machines and deployments.


## 9. Code Formatting (PEP 8)

PEP 8 is Python's official style guide. Consistent formatting reduces cognitive
load when reading and reviewing code.

Key rules: 4-space indentation, max line length (79-99 depending on convention),
consistent whitespace around operators, imports grouped and ordered, and
consistent naming (see Section 2.1).


In [18]:
# Before: inconsistent formatting

def   add(a,b) :
    result=a+b
    return result


In [19]:
# After: PEP 8 compliant

def add(a, b):
    result = a + b
    return result


### 9.1 Auto Formatters

- **Black** — opinionated, zero-config formatter
- **autopep8** — automatically reformats code to match PEP 8
- **isort** — sorts and groups import statements

```bash
pip install black autopep8 isort
black my_script.py
isort my_script.py
```


## 10. Linting

A linter analyzes code without running it to find unused imports, unused
variables, style violations, and likely bugs before they reach production.

**Common tools**

```bash
pip install flake8 pylint ruff

flake8 my_script.py
pylint my_script.py
ruff check my_script.py
```

Example of what a linter would flag:
```python
import os          # unused import
import json

def process(data):
    result = []    # never used
    return data
```
Running `flake8` on this file would report the unused `os` import and the unused
`result` variable.


## 11. Static Type Hints

Type hints document expected input and output types. They enable better editor
autocomplete, serve as inline documentation, and allow static type checkers
(e.g., `mypy`) to catch type errors before runtime.


In [20]:
from typing import List, Dict, Optional

def add(a: int, b: int) -> int:
    return a + b

def get_average(scores: List[float]) -> float:
    return sum(scores) / len(scores)

def find_user(users: Dict[str, int], name: str) -> Optional[int]:
    return users.get(name)

print(add(2, 3))
print(get_average([80.5, 90.0, 75.5]))
print(find_user({"alex": 101, "sam": 102}, "sam"))


5
82.0
102


## 12. Unit Testing Fundamentals

### 12.1 Why Testing?

Tests give confidence that code works as intended, and continues to work after
future changes. They catch regressions automatically instead of relying on
manual re-checking.

### 12.2 Testing Pyramid

```
        /\
       /  \      End-to-End Tests (few, slow, expensive)
      /----\
     /      \    Integration Tests (some)
    /--------\
   /          \  Unit Tests (many, fast, cheap)
  /------------\
```

### 12.3 Manual vs. Automated Testing

Manual testing is running the code and checking the output by hand each time.
Automated testing encodes those checks as repeatable code that runs in seconds
and can be executed on every change.


### 12.4 The `unittest` Module

`unittest` is Python's built-in testing framework, based on the `TestCase` class.


In [21]:
import unittest

def calculate_sum(a, b):
    return a + b

def is_valid_email(email: str) -> bool:
    return "@" in email and "." in email.split("@")[-1]

class TestCalculateSum(unittest.TestCase):

    def test_positive_numbers(self):
        self.assertEqual(calculate_sum(2, 3), 5)

    def test_negative_numbers(self):
        self.assertEqual(calculate_sum(-2, -3), -5)

    def test_returns_true_for_valid_email(self):
        self.assertTrue(is_valid_email("user@example.com"))

    def test_returns_false_for_invalid_email(self):
        self.assertFalse(is_valid_email("not-an-email"))

    def test_raises_on_invalid_input(self):
        with self.assertRaises(TypeError):
            calculate_sum("2", 3)

# Run tests inside a notebook (unittest.main() exits the interpreter by default)
suite = unittest.TestLoader().loadTestsFromTestCase(TestCalculateSum)
unittest.TextTestRunner(verbosity=2).run(suite)


test_negative_numbers (__main__.TestCalculateSum.test_negative_numbers) ... ok
test_positive_numbers (__main__.TestCalculateSum.test_positive_numbers) ... ok
test_raises_on_invalid_input (__main__.TestCalculateSum.test_raises_on_invalid_input) ... ok
test_returns_false_for_invalid_email (__main__.TestCalculateSum.test_returns_false_for_invalid_email) ... ok
test_returns_true_for_valid_email (__main__.TestCalculateSum.test_returns_true_for_valid_email) ... ok

----------------------------------------------------------------------
Ran 5 tests in 0.015s

OK


<unittest.runner.TextTestResult run=5 errors=0 failures=0>

## 13. Pytest

Pytest is a widely used third-party testing framework. It requires less
boilerplate than `unittest` and supports plain `assert` statements.

```bash
pip install pytest
```

Example test file `test_math_utils.py`:

```python
from math_utils import calculate_sum

def test_positive_numbers():
    assert calculate_sum(2, 3) == 5

def test_negative_numbers():
    assert calculate_sum(-2, -3) == -5
```

Run all tests in a project:

```bash
pytest
pytest -v                  # verbose output
pytest test_math_utils.py  # run a specific file
```

Pytest automatically discovers any file matching `test_*.py` or `*_test.py`,
and any function matching `test_*` inside it.


In [22]:
%%writefile test_math_utils_demo.py
def calculate_sum(a, b):
    return a + b

def test_positive_numbers():
    assert calculate_sum(2, 3) == 5

def test_negative_numbers():
    assert calculate_sum(-2, -3) == -5


Writing test_math_utils_demo.py


In [23]:
import subprocess
result = subprocess.run(["pytest", "test_math_utils_demo.py", "-v"], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)


============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: anyio-4.14.2, langsmith-0.10.2, typeguard-4.5.2
collecting ... collected 2 items

test_math_utils_demo.py::test_positive_numbers PASSED                    [ 50%]
test_math_utils_demo.py::test_negative_numbers PASSED                    [100%]

============================== 2 passed in 0.02s ===============================




### 13.1 Fixtures (Introduction)

Fixtures provide reusable setup/teardown logic for tests.

```python
import pytest

@pytest.fixture
def sample_documents():
    return ["doc one text", "doc two text", "doc three text"]

def test_document_count(sample_documents):
    assert len(sample_documents) == 3
```

### 13.2 Parameterized Tests (Introduction)

Parameterization runs the same test logic against multiple input/output pairs.

```python
import pytest

@pytest.mark.parametrize("a,b,expected", [
    (2, 3, 5),
    (-1, 1, 0),
    (0, 0, 0),
])
def test_calculate_sum(a, b, expected):
    assert calculate_sum(a, b) == expected
```


## 14. Mocking

### 14.1 Why Mock?

External dependencies (APIs, databases, file systems) are slow, costly, or
unavailable during automated testing. Mocking replaces them with controllable
stand-ins so tests run fast and deterministically.


In [24]:
from unittest.mock import patch, MagicMock

def get_completion(client, prompt: str) -> str:
    response = client.chat.completions.create(prompt=prompt)
    return response.choices[0].text

# Mock an OpenAI-style client without making a real API call
mock_client = MagicMock()
mock_client.chat.completions.create.return_value = MagicMock(
    choices=[MagicMock(text="This is a mocked model response")]
)

output = get_completion(mock_client, "Summarize this document")
print(output)
mock_client.chat.completions.create.assert_called_once_with(prompt="Summarize this document")


This is a mocked model response


In [25]:
# Mocking a database call

def fetch_user_record(db, user_id: int) -> dict:
    return db.query(user_id)

mock_db = MagicMock()
mock_db.query.return_value = {"id": 1, "name": "Test User"}

print(fetch_user_record(mock_db, 1))


{'id': 1, 'name': 'Test User'}


In [26]:
# Mocking file reading
# Designing the function to accept a reader dependency makes it directly
# testable without touching the real filesystem or patching builtins.

def load_prompt_template(path: str, reader=open) -> str:
    with reader(path) as f:
        return f.read()

mock_reader = MagicMock()
mock_reader.return_value.__enter__.return_value.read.return_value = "Summarize: {text}"

result = load_prompt_template("template.txt", reader=mock_reader)
print(result)
mock_reader.assert_called_once_with("template.txt")


Summarize: {text}


## 15. Testing AI Applications

AI pipelines have components that are well suited to unit testing even though the
final model output is non-deterministic: the deterministic logic *around* the model
call (preprocessing, chunking, similarity scoring) should always be tested directly,
and the model call itself should be mocked.


In [27]:
import re
import unittest

def clean_text(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

def chunk_text_by_words(text: str, chunk_size: int) -> list:
    words = text.split()
    return [
        " ".join(words[i:i + chunk_size])
        for i in range(0, len(words), chunk_size)
    ]

def cosine_similarity(vec_a: list, vec_b: list) -> float:
    dot = sum(a * b for a, b in zip(vec_a, vec_b))
    norm_a = sum(a ** 2 for a in vec_a) ** 0.5
    norm_b = sum(b ** 2 for b in vec_b) ** 0.5
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)

class TestTextPreprocessing(unittest.TestCase):

    def test_clean_text_lowercases_and_trims(self):
        self.assertEqual(clean_text("  Hello   WORLD  "), "hello world")

    def test_chunk_text_by_words(self):
        chunks = chunk_text_by_words("one two three four five", chunk_size=2)
        self.assertEqual(chunks, ["one two", "three four", "five"])

    def test_cosine_similarity_identical_vectors(self):
        self.assertAlmostEqual(cosine_similarity([1, 0], [1, 0]), 1.0)

    def test_cosine_similarity_orthogonal_vectors(self):
        self.assertAlmostEqual(cosine_similarity([1, 0], [0, 1]), 0.0)

    def test_cosine_similarity_zero_vector(self):
        self.assertEqual(cosine_similarity([0, 0], [1, 1]), 0.0)

suite = unittest.TestLoader().loadTestsFromTestCase(TestTextPreprocessing)
unittest.TextTestRunner(verbosity=2).run(suite)


test_chunk_text_by_words (__main__.TestTextPreprocessing.test_chunk_text_by_words) ... ok
test_clean_text_lowercases_and_trims (__main__.TestTextPreprocessing.test_clean_text_lowercases_and_trims) ... ok
test_cosine_similarity_identical_vectors (__main__.TestTextPreprocessing.test_cosine_similarity_identical_vectors) ... ok
test_cosine_similarity_orthogonal_vectors (__main__.TestTextPreprocessing.test_cosine_similarity_orthogonal_vectors) ... ok
test_cosine_similarity_zero_vector (__main__.TestTextPreprocessing.test_cosine_similarity_zero_vector) ... ok

----------------------------------------------------------------------
Ran 5 tests in 0.014s

OK


<unittest.runner.TextTestResult run=5 errors=0 failures=0>

In [28]:
# Testing a mocked RAG pipeline component

def retrieve_and_generate(query: str, retriever, llm_client) -> str:
    documents = retriever.search(query)
    context = " ".join(documents)
    return llm_client.generate(query, context)

mock_retriever = MagicMock()
mock_retriever.search.return_value = ["Paris is the capital of France."]

mock_llm = MagicMock()
mock_llm.generate.return_value = "The capital of France is Paris."

answer = retrieve_and_generate("What is the capital of France?", mock_retriever, mock_llm)
print(answer)
mock_retriever.search.assert_called_once_with("What is the capital of France?")


The capital of France is Paris.


## 16. Measuring Code Coverage

Code coverage measures the percentage of code executed by the test suite. High
coverage does not guarantee correctness, but low coverage reliably indicates
untested code paths.

```bash
pip install coverage

coverage run -m pytest
coverage report
coverage html   # generates an interactive HTML report
```

A typical report:

```
Name              Stmts   Miss  Cover
-------------------------------------
math_utils.py        12      1    92%
text_utils.py        20      6    70%
-------------------------------------
TOTAL                32      7    78%
```

The `Miss` column identifies exactly which lines were never executed during testing,
which is where new tests should be added first.


## 17. Performance Profiling

Before optimizing code, measure it. Profiling identifies real bottlenecks instead
of relying on assumptions.


In [29]:
import time

def slow_function(n: int) -> int:
    total = 0
    for i in range(n):
        total += i ** 2
    return total

start = time.time()
slow_function(1_000_000)
elapsed = time.time() - start
print(f"Elapsed time: {elapsed:.4f} seconds")


Elapsed time: 0.1063 seconds


In [30]:
import timeit

def sum_with_loop(n):
    total = 0
    for i in range(n):
        total += i
    return total

def sum_with_builtin(n):
    return sum(range(n))

loop_time = timeit.timeit(lambda: sum_with_loop(100_000), number=10)
builtin_time = timeit.timeit(lambda: sum_with_builtin(100_000), number=10)

print(f"Loop version:    {loop_time:.5f}s")
print(f"Built-in version: {builtin_time:.5f}s")


Loop version:    0.10698s
Built-in version: 0.04902s


In [31]:
import tracemalloc

def build_large_list(n: int) -> list:
    return [i for i in range(n)]

tracemalloc.start()
build_large_list(1_000_000)
current, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f"Current memory usage: {current / 1024:.2f} KB")
print(f"Peak memory usage:    {peak / 1024:.2f} KB")


Current memory usage: 0.96 KB
Peak memory usage:    39493.59 KB


## 18. Configuration Management

Configuration such as API keys, model names, and file paths should live outside
the codebase, typically in a `.env` file, and never be hardcoded or committed to
version control.

Example `.env` file:
```
OPENAI_API_KEY=sk-xxxxxxxxxxxxxxxx
MODEL_NAME=gpt-4o-mini
MAX_RETRIES=3
LOG_LEVEL=INFO
```

```bash
pip install python-dotenv
```


In [32]:
import os

# In a real project this would be: from dotenv import load_dotenv; load_dotenv()
# Simulated here since no .env file exists in this environment
os.environ.setdefault("MODEL_NAME", "gpt-4o-mini")
os.environ.setdefault("MAX_RETRIES", "3")

model_name = os.environ.get("MODEL_NAME")
max_retries = int(os.environ.get("MAX_RETRIES", 3))
api_key = os.environ.get("OPENAI_API_KEY")  # None if not set, never hardcoded

print(f"model_name={model_name}, max_retries={max_retries}, api_key_present={api_key is not None}")


model_name=gpt-4o-mini, max_retries=3, api_key_present=False


## 19. Project Structure for AI Applications

A production-oriented AI project typically separates concerns as follows:

```
app/
    config/
    services/
    prompts/
    utils/
    models/
    tests/
    logs/
    requirements.txt
    README.md
```

- `config/` — settings, environment loading
- `services/` — integrations with external APIs (LLM providers, vector stores)
- `prompts/` — prompt templates, versioned separately from logic
- `utils/` — reusable helper functions (chunking, cleaning, formatting)
- `models/` — data classes / schemas
- `tests/` — all unit and integration tests
- `logs/` — log output (typically excluded from version control)


## 20. Mini Hands-on Exercises

Complete each exercise in a separate cell below, or in your own project files.

1. Debug a broken calculator application.
2. Identify and fix common Python exceptions in a provided script.
3. Replace `print()` statements with structured logging.
4. Configure logging to both console and file simultaneously.
5. Write unit tests for a text preprocessing function.
6. Test a document chunking utility.
7. Validate CSV data using assertions.
8. Use the VS Code debugger to inspect variables and the call stack.
9. Lint and auto-format a poorly written Python script.
10. Refactor a monolithic script into reusable modules.


In [33]:
# Exercise 1: Debug a broken calculator application
# The function below has a bug. Find it, fix it, and add a test.

def calculator(a, b, operation):
    if operation == "add":
        return a + b
    elif operation == "subtract":
        return a - b
    elif operation == "multiply":
        return a * b
    elif operation == "divide":
        return a / b  # bug: no protection against division by zero

# Your task:
# 1. Reproduce the bug by calling calculator(5, 0, "divide")
# 2. Fix it using proper exception handling
# 3. Write a unit test that asserts a ZeroDivisionError is handled gracefully


In [34]:
# Exercise 5-6: Write unit tests for preprocessing and chunking
# Use TestTextPreprocessing from Section 15 as a reference.
# Add at least one new test case covering an edge case (e.g., empty string input).


In [35]:
# Exercise 7: Validate CSV-like data using assertions

def validate_row(row: dict) -> None:
    assert "name" in row, "Missing 'name' field"
    assert "age" in row, "Missing 'age' field"
    assert isinstance(row["age"], int), "'age' must be an integer"
    assert row["age"] >= 0, "'age' must not be negative"

sample_rows = [
    {"name": "Alex", "age": 30},
    {"name": "Sam", "age": -5},
]

for row in sample_rows:
    try:
        validate_row(row)
        print(f"Valid row: {row}")
    except AssertionError as e:
        print(f"Invalid row {row}: {e}")


Valid row: {'name': 'Alex', 'age': 30}
Invalid row {'name': 'Sam', 'age': -5}: 'age' must not be negative


## 21. Capstone Mini Project: Production-Ready Document Processing Pipeline

Build a small pipeline that demonstrates every practice covered in this module:

- Read documents (TXT/CSV/PDF)
- Clean and preprocess text
- Perform document chunking
- Log each processing step with appropriate log levels
- Handle invalid files using custom exceptions
- Store configuration via environment variables
- Organize the project into modules (represented here as functions/classes)
- Include unit tests for preprocessing and chunking
- Follow PEP 8 and pass linting/formatting checks

The full working code is below. In a real project each section would live in its
own module (`config.py`, `exceptions.py`, `pipeline.py`, `tests/test_pipeline.py`).


In [36]:
import os
import re
import logging
import unittest

# --- config.py equivalent ---

CHUNK_SIZE = int(os.environ.get("CHUNK_SIZE", 20))
LOG_LEVEL = os.environ.get("LOG_LEVEL", "INFO")
SUPPORTED_EXTENSIONS = (".txt", ".csv")


In [37]:
# --- exceptions.py equivalent ---

class DocumentPipelineError(Exception):
    """Base exception for the document processing pipeline."""
    pass

class UnsupportedFileTypeError(DocumentPipelineError):
    """Raised when a file extension is not supported."""
    pass

class EmptyDocumentError(DocumentPipelineError):
    """Raised when a document has no usable content."""
    pass


In [38]:
# --- logging setup ---

pipeline_logger = logging.getLogger("document_pipeline")
pipeline_logger.setLevel(getattr(logging, LOG_LEVEL, logging.INFO))

if not pipeline_logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(
        logging.Formatter("%(asctime)s | %(levelname)s | %(name)s | %(message)s")
    )
    pipeline_logger.addHandler(handler)


In [39]:
# --- pipeline.py equivalent ---

def check_file_type(filename: str) -> None:
    if not filename.lower().endswith(SUPPORTED_EXTENSIONS):
        pipeline_logger.error(f"Unsupported file type: {filename}")
        raise UnsupportedFileTypeError(f"'{filename}' is not a supported file type")
    pipeline_logger.info(f"File type accepted: {filename}")


def clean_text(text: str) -> str:
    pipeline_logger.debug("Cleaning text")
    cleaned = text.lower().strip()
    cleaned = re.sub(r"\s+", " ", cleaned)
    return cleaned


def chunk_document(text: str, chunk_size: int = CHUNK_SIZE) -> list:
    if not text:
        pipeline_logger.error("Attempted to chunk empty document")
        raise EmptyDocumentError("Document has no content to chunk")

    words = text.split()
    chunks = [
        " ".join(words[i:i + chunk_size])
        for i in range(0, len(words), chunk_size)
    ]
    pipeline_logger.info(f"Document split into {len(chunks)} chunks")
    return chunks


def process_document(filename: str, raw_text: str) -> list:
    pipeline_logger.info(f"Starting processing for {filename}")
    check_file_type(filename)
    cleaned = clean_text(raw_text)
    chunks = chunk_document(cleaned)
    pipeline_logger.info(f"Finished processing for {filename}")
    return chunks


In [40]:
# --- demonstration run ---

sample_text = (
    "GenAI pipelines require careful preprocessing before embedding generation. "
    "Chunking breaks long documents into smaller, retrievable pieces of context "
    "that fit within a model's context window while preserving meaning."
)

result_chunks = process_document("quarterly_report.txt", sample_text)
for idx, chunk in enumerate(result_chunks, start=1):
    print(f"Chunk {idx}: {chunk}")


2026-08-03 07:47:17,656 | INFO | document_pipeline | Starting processing for quarterly_report.txt
INFO:document_pipeline:Starting processing for quarterly_report.txt
2026-08-03 07:47:17,658 | INFO | document_pipeline | File type accepted: quarterly_report.txt
INFO:document_pipeline:File type accepted: quarterly_report.txt
2026-08-03 07:47:17,660 | INFO | document_pipeline | Document split into 2 chunks
INFO:document_pipeline:Document split into 2 chunks
2026-08-03 07:47:17,662 | INFO | document_pipeline | Finished processing for quarterly_report.txt
INFO:document_pipeline:Finished processing for quarterly_report.txt


Chunk 1: genai pipelines require careful preprocessing before embedding generation. chunking breaks long documents into smaller, retrievable pieces of context that fit
Chunk 2: within a model's context window while preserving meaning.


In [41]:
# --- error path demonstration ---

try:
    process_document("malware.exe", sample_text)
except UnsupportedFileTypeError as e:
    print(f"Handled error: {e}")

try:
    process_document("empty.txt", "")
except EmptyDocumentError as e:
    print(f"Handled error: {e}")


2026-08-03 07:47:18,508 | INFO | document_pipeline | Starting processing for malware.exe
INFO:document_pipeline:Starting processing for malware.exe
2026-08-03 07:47:18,511 | ERROR | document_pipeline | Unsupported file type: malware.exe
ERROR:document_pipeline:Unsupported file type: malware.exe
2026-08-03 07:47:18,513 | INFO | document_pipeline | Starting processing for empty.txt
INFO:document_pipeline:Starting processing for empty.txt
2026-08-03 07:47:18,515 | INFO | document_pipeline | File type accepted: empty.txt
INFO:document_pipeline:File type accepted: empty.txt
2026-08-03 07:47:18,516 | ERROR | document_pipeline | Attempted to chunk empty document
ERROR:document_pipeline:Attempted to chunk empty document


Handled error: 'malware.exe' is not a supported file type
Handled error: Document has no content to chunk


In [42]:
# --- tests/test_pipeline.py equivalent ---

class TestDocumentPipeline(unittest.TestCase):

    def test_check_file_type_accepts_supported_extension(self):
        try:
            check_file_type("notes.txt")
        except UnsupportedFileTypeError:
            self.fail("check_file_type raised unexpectedly for a supported extension")

    def test_check_file_type_rejects_unsupported_extension(self):
        with self.assertRaises(UnsupportedFileTypeError):
            check_file_type("archive.zip")

    def test_clean_text_normalizes_whitespace(self):
        self.assertEqual(clean_text("  Hello   World  "), "hello world")

    def test_chunk_document_splits_correctly(self):
        text = " ".join(str(i) for i in range(45))
        chunks = chunk_document(text, chunk_size=20)
        self.assertEqual(len(chunks), 3)
        self.assertEqual(len(chunks[0].split()), 20)

    def test_chunk_document_raises_on_empty_input(self):
        with self.assertRaises(EmptyDocumentError):
            chunk_document("")

    def test_process_document_end_to_end(self):
        chunks = process_document("report.txt", "one two three four five")
        self.assertIsInstance(chunks, list)
        self.assertTrue(len(chunks) > 0)


suite = unittest.TestLoader().loadTestsFromTestCase(TestDocumentPipeline)
unittest.TextTestRunner(verbosity=2).run(suite)


test_check_file_type_accepts_supported_extension (__main__.TestDocumentPipeline.test_check_file_type_accepts_supported_extension) ... 2026-08-03 07:47:19,895 | INFO | document_pipeline | File type accepted: notes.txt
INFO:document_pipeline:File type accepted: notes.txt
ok
test_check_file_type_rejects_unsupported_extension (__main__.TestDocumentPipeline.test_check_file_type_rejects_unsupported_extension) ... 2026-08-03 07:47:19,900 | ERROR | document_pipeline | Unsupported file type: archive.zip
ERROR:document_pipeline:Unsupported file type: archive.zip
ok
test_chunk_document_raises_on_empty_input (__main__.TestDocumentPipeline.test_chunk_document_raises_on_empty_input) ... 2026-08-03 07:47:19,907 | ERROR | document_pipeline | Attempted to chunk empty document
ERROR:document_pipeline:Attempted to chunk empty document
ok
test_chunk_document_splits_correctly (__main__.TestDocumentPipeline.test_chunk_document_splits_correctly) ... 2026-08-03 07:47:19,916 | INFO | document_pipeline | Docume

<unittest.runner.TextTestResult run=6 errors=0 failures=0>

In [43]:
# --- requirements.txt equivalent ---

requirements_txt = """python-dotenv==1.0.1
pytest==8.2.0
coverage==7.5.0
black==24.4.0
flake8==7.0.0
"""

with open("requirements.txt", "w") as f:
    f.write(requirements_txt)

print(requirements_txt)


python-dotenv==1.0.1
pytest==8.2.0
coverage==7.5.0
black==24.4.0
flake8==7.0.0



### Capstone Checklist

- [x] Reads and validates document input (extension check)
- [x] Cleans and normalizes text
- [x] Chunks documents into retrievable pieces
- [x] Logs each step at an appropriate level (`DEBUG`, `INFO`, `ERROR`)
- [x] Raises and handles custom exceptions for invalid input
- [x] Reads configuration from environment variables
- [x] Organized into logical, single-purpose functions
- [x] Covered by unit tests, including edge cases
- [x] `requirements.txt` generated for reproducibility

This pipeline is the foundation for the next modules: NumPy, Pandas, text
preprocessing at scale, embeddings, and full RAG pipelines.
